# Barcelona Tourism Pressure Forecasting
This notebook analyzes Airbnb data to **quantify and forecast** "tourism pressure" across different neighborhoods in Barcelona. It combines geospatial data with active listings and future availability calendars to build a composite Tourism Pressure Index. This can significantly help in urban planning.

In [12]:
#Import necessary libraries for data manipulation, geospatial analysis, and geometric parsing
import pandas as pd
import geopandas as gpd
from shapely import wkt

## 1. Data Loading & Exploration
We load the neighborhood boundaries (polygons) and the Airbnb datasets (listings and availability calendar).

In [13]:
barris_raw = pd.read_csv("BarcelonaCiutat_Barris.csv")
barris = gpd.GeoDataFrame(
    barris_raw,
    geometry=gpd.GeoSeries.from_wkt(barris_raw["geometria_wgs84"], on_invalid='ignore'),
    crs="EPSG:4326"
)

listings = pd.read_csv("listings.csv.gz", compression="gzip", low_memory=False)
calendar = pd.read_csv("calendar.csv.gz", compression="gzip", low_memory=False)

print(listings.columns.tolist())
print(calendar.columns.tolist())

['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minim

## 2. Feature Extraction & Data Cleaning
We isolate key properties from the listings dataset and clean the `price` column by removing currency strings and converting it to a float.

In [14]:
# Keep only features relevant to location, pricing, capacity, and booking activity
cols = ["id", "neighbourhood_cleansed", "latitude", "longitude",
        "room_type", "price", "availability_365", "number_of_reviews",
        "reviews_per_month"]
listings = listings[cols].copy()

# Strip whitespace and formatting commas from the price string to cast it to a numerical type
listings["price"] = listings["price"].replace(r'[\s,]','', regex=True).astype(float)

# Preview the top 10 neighborhoods with the highest absolute count of listings
print(listings["neighbourhood_cleansed"].value_counts().head(10))

neighbourhood_cleansed
la Dreta de l'Eixample                   2404
el Raval                                 1330
el Barri Gòtic                           1167
Sant Pere, Santa Caterina i la Ribera    1141
la Vila de Gràcia                        1056
la Sagrada Família                       1047
l'Antiga Esquerra de l'Eixample          1006
Sant Antoni                               947
el Poble Sec                              823
la Nova Esquerra de l'Eixample            766
Name: count, dtype: int64


## 3. Geospatial Analysis: Listing Density per Neighborhood
We map individual listings (points) into neighborhood boundaries (polygons) using a spatial join, then calculate listings per $km^2$.

In [15]:
# Convert the flat listings DataFrame into a GeoDataFrame using its coordinates
gdf_listings = gpd.GeoDataFrame(
    listings,
    geometry=gpd.points_from_xy(listings["longitude"], listings["latitude"]),
    crs="EPSG:4326"
)

# Perform a spatial join to assign neighborhood IDs ('codi_barri', 'nom_barri') to each listing point
listings_joined = gpd.sjoin(gdf_listings, barris[["codi_barri", "nom_barri", "geometry"]], how="left", predicate="within")

# Group by neighborhood to calculate total active listings
density = listings_joined.groupby('nom_barri').agg(
    total_listings=("id", "count")
).reset_index()

# Project neighborhood geometries to a metric CRS (EPSG:25831 - UTM zone 31N) to accurately compute area in square kilometers
barris["area_km2"] = barris.geometry.to_crs("EPSG:25831").area / 1e6

# Merge calculated areas back and compute listing concentration (density)
density = density.merge(barris[["nom_barri", "area_km2"]], on="nom_barri")
density["listings_per_km2"] = density["total_listings"] / density["area_km2"]

print(density.sort_values("listings_per_km2", ascending=False).head(10))

                                nom_barri  total_listings  area_km2  \
27                         el Barri Gòtic            1166  0.815595   
42                               el Raval            1340  1.100291   
13                            Sant Antoni             943  0.804114   
48                 la Dreta de l'Eixample            2404  2.119881   
18  Sant Pere, Santa Caterina i la Ribera            1138  1.109653   
56                     la Sagrada Família            1046  1.041791   
44        l'Antiga Esquerra de l'Eixample            1001  1.228455   
65                      la Vila de Gràcia            1056  1.321182   
54         la Nova Esquerra de l'Eixample             766  1.340677   
36                          el Fort Pienc             490  0.929367   

    listings_per_km2  
27       1429.631022  
42       1217.859433  
13       1172.719877  
48       1134.026091  
18       1025.545858  
56       1004.040359  
44        814.844494  
65        799.284253  
54        5

## 4. Temporal Analysis: Monthly Occupancy Rates
Using the forward-looking calendar dataset, we derive monthly occupancy rates. In Airbnb data, `available = 'f'` implies the night is blocked or booked.

In [17]:
# Reload fresh
calendar = pd.read_csv("calendar.csv.gz", compression="gzip", low_memory=False)

# Clean
calendar["date"] = pd.to_datetime(calendar["date"])
calendar["available"] = calendar["available"].map({"t": 1, "f": 0})
calendar["listing_id"] = calendar["listing_id"].astype(str)

# Ensure 'id' column in listings_joined is also string type for merge compatibility
listings_joined["id"] = listings_joined["id"].astype(str)

# Merge neighbourhood from listings_joined
calendar = calendar.merge(
    listings_joined[["id", "nom_barri"]].drop_duplicates("id"),
    left_on="listing_id", right_on="id",
    how="left"
).dropna(subset=["nom_barri"])

# Monthly occupancy per neighbourhood
calendar["year_month"] = calendar["date"].dt.to_period("M")

occupancy = calendar.groupby(["nom_barri", "year_month"]).agg(
    occupancy_rate=("available", lambda x: 1 - x.mean())
).reset_index()

print(occupancy.shape)
print(occupancy.dropna().head(30))

(897, 3)
      nom_barri year_month  occupancy_rate
0      Can Baró    2025-12        0.641958
1      Can Baró    2026-01        0.438710
2      Can Baró    2026-02        0.372321
3      Can Baró    2026-03        0.324194
4      Can Baró    2026-04        0.280000
5      Can Baró    2026-05        0.253226
6      Can Baró    2026-06        0.305833
7      Can Baró    2026-07        0.330645
8      Can Baró    2026-08        0.325000
9      Can Baró    2026-09        0.500000
10     Can Baró    2026-10        0.600000
11     Can Baró    2026-11        0.600000
12     Can Baró    2026-12        0.634286
13  Can Peguera    2025-12        0.542857
14  Can Peguera    2026-01        0.500000
15  Can Peguera    2026-02        0.500000
16  Can Peguera    2026-03        0.500000
17  Can Peguera    2026-04        0.500000
18  Can Peguera    2026-05        0.500000
19  Can Peguera    2026-06        0.500000
20  Can Peguera    2026-07        0.500000
21  Can Peguera    2026-08        0.500000
22

## 5. Constructing the Tourism Pressure Index
We merge our static asset density metric with the latest temporal occupancy metric. Both metrics are scaled to a $[0, 1]$ range and combined via a weighted formula:

$$\text{Pressure Index} = (0.60 \times \text{Scaled Density}) + (0.40 \times \text{Scaled Occupancy})$$

In [18]:
from sklearn.preprocessing import MinMaxScaler

# Filter for the most recent month in the dataset to build a current snapshot of pressure
pressure = density.merge(
    occupancy[occupancy["year_month"] == occupancy["year_month"].max()],
    on="nom_barri"
).dropna(subset=["listings_per_km2", "occupancy_rate"])

# Initialize MinMaxScaler to put both metrics on an equal playing field before weighting
scaler = MinMaxScaler()
features = ["listings_per_km2", "occupancy_rate"]
pressure_scaled = pressure.copy()
pressure_scaled[features] = scaler.fit_transform(pressure[features])

# Apply the custom weighted formula to establish the final metric
pressure_scaled["pressure_index"] = (
    0.60 * pressure_scaled["listings_per_km2"] +
    0.40 * pressure_scaled["occupancy_rate"]
)

# Output final rankings of neighborhoods experiencing the highest tourism pressure
print(pressure_scaled[["nom_barri", "listings_per_km2", "occupancy_rate", "pressure_index"]]
      .sort_values("pressure_index", ascending=False)
      .head(30))

                                nom_barri  listings_per_km2  occupancy_rate  \
27                         el Barri Gòtic          1.000000        0.334183   
42                               el Raval          0.851739        0.365371   
13                            Sant Antoni          0.820136        0.232158   
48                 la Dreta de l'Eixample          0.793047        0.233383   
18  Sant Pere, Santa Caterina i la Ribera          0.717100        0.336024   
56                     la Sagrada Família          0.702044        0.298230   
44        l'Antiga Esquerra de l'Eixample          0.569587        0.326405   
65                      la Vila de Gràcia          0.558694        0.341606   
3                        Ciutat Meridiana          0.002831        1.000000   
2                               Canyelles          0.000000        1.000000   
30     el Camp d'en Grassot i Gràcia Nova          0.317667        0.454116   
54         la Nova Esquerra de l'Eixample          0

## 6. Time-Series Forecasting
We isolate a single neighborhood to model future occupancy trends.
> **Note:** Because the standard dataset tracks 12–13 months forward, there are not enough historic periods to fit a seasonal baseline ($2 \times \text{seasonal periods}$). Thus, we drop seasonality and use a linear Holt-Winters trend model.

In [19]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Pick one neighbourhood to demonstrate
barri_name = "les Corts"
ts = occupancy[occupancy["nom_barri"] == barri_name].set_index("year_month")["occupancy_rate"]
ts.index = ts.index.to_timestamp()

print(f"Number of data points for '{barri_name}': {len(ts)}")

# Fit model
# The error indicates that there are not enough data points (less than 2 * seasonal_periods) to estimate the seasonal component. Setting seasonal=None to proceed.
model = ExponentialSmoothing(ts, trend="add", seasonal=None, seasonal_periods=12)
fit = model.fit()

# Forecast 6 months ahead - for example
forecast = fit.forecast(6)
print(forecast)

Number of data points for 'les Corts': 13
2027-01-01    0.619110
2027-02-01    0.654644
2027-03-01    0.690178
2027-04-01    0.725712
2027-05-01    0.761246
2027-06-01    0.796781
Freq: MS, dtype: float64


## 7. Geospatial Visualization
Finally, we attach our engineered index back to the geographic polygons and generate an interactive Folium choropleth map saved as an HTML artifact.

In [21]:
import folium

#Join calculated pressure index back into the geographic boundary datasets
pressure_map = barris.merge(pressure_scaled[["nom_barri", "pressure_index"]], on="nom_barri")

# Initialize an interactive map centered over the coordinates of Barcelona
m = folium.Map(location=[41.39, 2.15], zoom_start=12)

folium.Choropleth(
    geo_data=pressure_map.to_json(), # Convert GeoDataFrame to GeoJSON string for robust rendering
    data=pressure_map,
    columns=["nom_barri", "pressure_index"],
    key_on="feature.properties.nom_barri",
    fill_color="YlOrRd",
    legend_name="Tourism Pressure Index"
).add_to(m)

m.save("barcelona_tourism_pressure.html")